In [1]:
import pandas as pd
import numpy as np
import json
import time
import asyncio
from typing import List, Tuple
import sys
import os

# Add the parent directory to the path to import modules
sys.path.append('..')

# Install nest_asyncio to handle async in Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("Applied nest_asyncio for Jupyter compatibility")
except ImportError:
    print("Installing nest_asyncio...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio"])
    import nest_asyncio
    nest_asyncio.apply()
    print("Installed and applied nest_asyncio")

from oracle.adversarial_queries import AdversarialQueryOracle, AdversarialQueryInput, get_predicate_graph
from workload.workloads import get_workload_set
from logger.log import l
from oracle.pg_celery_worker.pg_worker import contracts


Applied nest_asyncio for Jupyter compatibility


In [2]:
# Load the experimental results
df = pd.read_csv('best_results.csv')
print(f"Loaded {len(df)} experimental results")
print("\nFirst few rows:")
print(df.head())

# Parse the train_x column to extract query and plan
def parse_train_x(train_x_str: str) -> Tuple[str, List[int]]:
    """Parse train_x string to extract query and plan"""
    parts = train_x_str.split('[SEP]')
    if len(parts) != 2:
        raise ValueError(f"Expected 2 parts separated by [SEP], got {len(parts)}")
    
    query_str = parts[0]
    plan_str = parts[1]
    
    # Convert plan string to list of integers
    plan_list = [int(x) for x in plan_str.split(',')]
    
    return query_str, plan_list

# Test parsing on first row
first_row = df.iloc[0]
query, plan = parse_train_x(first_row['train_x'])
print(f"\nExample parsing:")
print(f"Query: {query}")
print(f"Plan: {plan}")
print(f"Runtime ratio (train_y): {first_row['train_y']}")


Loaded 40 experimental results

First few rows:
                                             train_x    train_y  censoring
0  (aka_title (season_nr = first))(company_type )...  12.837399        0.0
1  (aka_title (imdb_index most_popular)(season_nr...   7.620608        1.0
2  (aka_name )(char_name (imdb_index random))(com...   3.165690        0.0
3  (aka_name (imdb_index most_popular))(kind_type...   2.740368        0.0
4  (aka_name )(company_type (kind random))(link_t...   2.347878        0.0

Example parsing:
Query: (aka_title (season_nr = first))(company_type )(complete_cast )(link_type )(movie_info )(movie_info_idx )(person_info )(title )
Plan: [1, 1, 20, 1, 20, 12, 20, 10, 20, 2, 20, 6, 20, 6, 20, 9, 20, 6, 20, 15, 20, 2, 20, 15, 20, 17]
Runtime ratio (train_y): 12.837398901816178


In [5]:
# Set up the adversarial query oracle
print("Setting up adversarial query oracle...")
predicate_graph = get_predicate_graph(get_workload_set("JOB"))
oracle = AdversarialQueryOracle(predicate_graph)
print("Oracle setup complete!")

def run_query_multiple_times(oracle: AdversarialQueryOracle, 
                            query_str: str, 
                            plan: List[int] = None, 
                            num_runs: int = 5,
                            timeout_ms: int = 30000) -> Tuple[float, float]:
    runtimes = []
    
    inputs = [AdversarialQueryInput(llm_output=query_str, plan=plan, timeout_ms=timeout_ms)] * num_runs
    results = oracle.query(inputs)
    assert all(isinstance(result.result, contracts.QueryCompleteResponse) for result in results)
    runtimes = [result.result.elapsed_secs for result in results]

    mean_time = np.mean(runtimes)
    std_time = np.std(runtimes)
    
    return mean_time, std_time


Setting up adversarial query oracle...
Oracle setup complete!


In [7]:
# Test with the first query
print(f"\nTesting query execution...")
query, plan = parse_train_x(first_row['train_x'])

print("Running default plan...")
default_mean, default_std = run_query_multiple_times(oracle, query, plan=None, num_runs=3, timeout_ms=15000)
print(f"Default plan: {default_mean:.3f} ± {default_std:.3f} seconds")

print("\nRunning bayes-optimized plan...")
bayes_mean, bayes_std = run_query_multiple_times(oracle, query, plan=plan, num_runs=3, timeout_ms=15000)
print(f"Bayes-optimized plan: {bayes_mean:.3f} ± {bayes_std:.3f} seconds")

new_ratio = default_mean / bayes_mean
experimental_ratio = first_row['train_y']
print(f"\nResults:")
print(f"New ratio: {new_ratio:.3f}")
print(f"Experimental ratio: {experimental_ratio:.3f}")
print(f"Difference: {abs(new_ratio - experimental_ratio):.3f}")



Testing query execution...
Running default plan...
16:17:42.316 | INFO     | adversarial_queries:query:157 - 🚀 Starting batch processing of 3 queries
16:17:42.317 | INFO     | adversarial_queries:__query_batch:168 - ⚡ Dispatching 3 async tasks
16:17:42.317 | INFO     | adversarial_queries:__query_one:178 - 🔵 Task 0: Starting query processing
16:17:42.321 | INFO     | adversarial_queries:__query_one:178 - 🔵 Task 1: Starting query processing
16:17:42.324 | INFO     | adversarial_queries:__query_one:178 - 🔵 Task 2: Starting query processing
16:17:42.641 | INFO     | adversarial_queries:submit:448 - Submitted job ID 438493
16:17:42.801 | INFO     | adversarial_queries:submit:448 - Submitted job ID 438494
16:17:42.966 | INFO     | adversarial_queries:submit:448 - Submitted job ID 438495
16:17:43.424 | INFO     | adversarial_queries:__query_one:221 - ✅ Task 0: Completed in 1.11s
16:17:43.570 | INFO     | adversarial_queries:__query_one:221 - ✅ Task 1: Completed in 1.25s
16:17:43.757 | INFO 

In [8]:
def validate_all_queries(df: pd.DataFrame, 
                         oracle: AdversarialQueryOracle, 
                         num_runs: int = 5,
                         timeout_ms: int = 30000,
                         max_queries: int = None) -> pd.DataFrame:
    """
    Validate all queries in the dataframe by running them multiple times
    and comparing the results to experimental data
    
    Args:
        df: DataFrame with experimental results
        oracle: AdversarialQueryOracle instance
        num_runs: Number of times to run each query variant
        timeout_ms: Timeout for each query in milliseconds
        max_queries: Maximum number of queries to process (for testing)
    
    Returns:
        DataFrame with validation results
    """
    results = []
    
    # Limit number of queries for testing if specified
    queries_to_process = df.head(max_queries) if max_queries else df
    
    for idx, row in queries_to_process.iterrows():
        print(f"\nProcessing query {idx + 1}/{len(queries_to_process)}...")
        
        try:
            # Parse the query and plan
            query_str, bayes_plan = parse_train_x(row['train_x'])
            experimental_ratio = row['train_y']
            
            print(f"Query: {query_str[:100]}...")  # Print first 100 chars
            print(f"Plan: {bayes_plan}")
            print(f"Experimental ratio: {experimental_ratio:.3f}")
            
            # Run default plan multiple times
            print(f"Running default plan {num_runs} times...")
            default_mean, default_std = run_query_multiple_times(
                oracle, query_str, plan=None, num_runs=num_runs, timeout_ms=timeout_ms
            )
            
            # Run bayes-optimized plan multiple times
            print(f"Running bayes-optimized plan {num_runs} times...")
            bayes_mean, bayes_std = run_query_multiple_times(
                oracle, query_str, plan=bayes_plan, num_runs=num_runs, timeout_ms=timeout_ms
            )
            
            # Calculate new ratio
            new_ratio = default_mean / bayes_mean if bayes_mean > 0 else float('inf')
            ratio_difference = abs(new_ratio - experimental_ratio)
            
            print(f"Default: {default_mean:.3f} ± {default_std:.3f}s")
            print(f"Bayes: {bayes_mean:.3f} ± {bayes_std:.3f}s")
            print(f"New ratio: {new_ratio:.3f}")
            print(f"Difference from experimental: {ratio_difference:.3f}")
            
            # Store results
            results.append({
                'query_idx': idx,
                'query': query_str,
                'bayes_plan': str(bayes_plan),
                'default_mean_runtime': default_mean,
                'default_std_runtime': default_std,
                'bayes_mean_runtime': bayes_mean,
                'bayes_std_runtime': bayes_std,
                'experimental_ratio': experimental_ratio,
                'new_ratio': new_ratio,
                'ratio_difference': ratio_difference,
                'censoring': row['censoring']
            })
            
        except Exception as e:
            print(f"Error processing query {idx}: {e}")
            # Store error result
            results.append({
                'query_idx': idx,
                'query': f"ERROR: {str(e)}",
                'bayes_plan': "N/A",
                'default_mean_runtime': float('nan'),
                'default_std_runtime': float('nan'),
                'bayes_mean_runtime': float('nan'),
                'bayes_std_runtime': float('nan'),
                'experimental_ratio': row['train_y'],
                'new_ratio': float('nan'),
                'ratio_difference': float('nan'),
                'censoring': row['censoring']
            })
    
    return pd.DataFrame(results)

# Run validation on a subset first (first 3 queries for testing)
print("Running validation on first 3 queries...")
validation_results = validate_all_queries(df, oracle, num_runs=3, timeout_ms=15000)


Running validation on first 3 queries...

Processing query 1/40...
Query: (aka_title (season_nr = first))(company_type )(complete_cast )(link_type )(movie_info )(movie_info_i...
Plan: [1, 1, 20, 1, 20, 12, 20, 10, 20, 2, 20, 6, 20, 6, 20, 9, 20, 6, 20, 15, 20, 2, 20, 15, 20, 17]
Experimental ratio: 12.837
Running default plan 3 times...
16:18:00.963 | INFO     | adversarial_queries:query:157 - 🚀 Starting batch processing of 3 queries
16:18:00.965 | INFO     | adversarial_queries:__query_batch:168 - ⚡ Dispatching 3 async tasks
16:18:00.965 | INFO     | adversarial_queries:__query_one:178 - 🔵 Task 0: Starting query processing
16:18:00.968 | INFO     | adversarial_queries:__query_one:178 - 🔵 Task 1: Starting query processing
16:18:00.970 | INFO     | adversarial_queries:__query_one:178 - 🔵 Task 2: Starting query processing
16:18:01.116 | INFO     | adversarial_queries:submit:448 - Submitted job ID 438617
16:18:01.278 | INFO     | adversarial_queries:submit:448 - Submitted job ID 438618
16

In [9]:
# Display the validation results
print("Validation Results Summary:")
print("=" * 80)

# Display detailed results
for idx, row in validation_results.iterrows():
    print(f"\nQuery {row['query_idx']}:")
    print(f"  Experimental ratio: {row['experimental_ratio']:.3f}")
    print(f"  New ratio: {row['new_ratio']:.3f}")
    print(f"  Difference: {row['ratio_difference']:.3f}")
    print(f"  Default runtime: {row['default_mean_runtime']:.3f} ± {row['default_std_runtime']:.3f}s")
    print(f"  Bayes runtime: {row['bayes_mean_runtime']:.3f} ± {row['bayes_std_runtime']:.3f}s")

# Summary statistics
print(f"\nSummary Statistics:")
print(f"Number of queries processed: {len(validation_results)}")
print(f"Average ratio difference: {validation_results['ratio_difference'].mean():.3f}")
print(f"Median ratio difference: {validation_results['ratio_difference'].median():.3f}")
print(f"Max ratio difference: {validation_results['ratio_difference'].max():.3f}")

# Display the results dataframe
print(f"\nDetailed Results DataFrame:")
display(validation_results[['query_idx', 'experimental_ratio', 'new_ratio', 
                          'ratio_difference', 'default_mean_runtime', 'bayes_mean_runtime']])


Validation Results Summary:

Query 0:
  Experimental ratio: 12.837
  New ratio: 0.331
  Difference: 12.506
  Default runtime: 0.074 ± 0.003s
  Bayes runtime: 0.223 ± 0.002s

Query 1:
  Experimental ratio: 7.621
  New ratio: 0.151
  Difference: 7.470
  Default runtime: 0.193 ± 0.003s
  Bayes runtime: 1.280 ± 0.005s

Query 2:
  Experimental ratio: 3.166
  New ratio: 1.427
  Difference: 1.738
  Default runtime: 0.711 ± 0.008s
  Bayes runtime: 0.498 ± 0.005s

Query 3:
  Experimental ratio: 2.740
  New ratio: 1.870
  Difference: 0.870
  Default runtime: 0.689 ± 0.023s
  Bayes runtime: 0.369 ± 0.032s

Query 4:
  Experimental ratio: 2.348
  New ratio: 0.549
  Difference: 1.799
  Default runtime: 0.456 ± 0.179s
  Bayes runtime: 0.831 ± 0.007s

Query 5:
  Experimental ratio: 1.890
  New ratio: nan
  Difference: nan
  Default runtime: nan ± nans
  Bayes runtime: nan ± nans

Query 6:
  Experimental ratio: 1.886
  New ratio: 0.872
  Difference: 1.014
  Default runtime: 0.396 ± 0.003s
  Bayes runti

,query_idx,experimental_ratio,new_ratio,ratio_difference,default_mean_runtime,bayes_mean_runtime
0,0,12.837399,0.331103,12.506296,0.073831,0.222986
1,1,7.620608,0.150660,7.469948,0.192831,1.279914
2,2,3.165690,1.427256,1.738433,0.711468,0.498486
3,3,2.740368,1.870012,0.870357,0.689341,0.368629
4,4,2.347878,0.548520,1.799358,0.456064,0.831444
5,5,1.889819,NaN,NaN,NaN,NaN
6,6,1.886015,0.871836,1.014179,0.395599,0.453754
7,7,1.870387,1.861706,0.008680,0.908755,0.488130
8,8,1.867723,1.889094,0.021371,0.925464,0.489898
9,9,1.833169,0.083982,1.749186,0.011441,0.136225
